In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
#from robustness_experiment_box.analysis.report_creator import ReportCreator
import re
import scipy.stats as st


In [ ]:
df = pd.read_csv('mnist_full_results.csv')
print(df.columns)

metrics = ['IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility', 'morans_i_difference','morans_i_explanation', 'optim_time']
data = "MNIST"

Index(['Unnamed: 0', 'network', 'image', 'original_label',
       'original_predicted_label', 'predicted_label', 'target', 'method',
       'IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility',
       'morans_i_original', 'morans_i_explanation',
       'morans_i_original_minus_explanation', 'optim_time',
       'correctness_ratio', 'weak_correctness', 'morans_i_difference',
       'timeout'],
      dtype='object')


In [14]:
df.head()

,Unnamed: 0,network,image,original_label,original_predicted_label,predicted_label,target,method,IM1,IM2,...,linf,implausibility,morans_i_original,morans_i_explanation,morans_i_original_minus_explanation,optim_time,correctness_ratio,weak_correctness,morans_i_difference,timeout
0,0,mnist_output_100,0.0,7,7,NaN,0.0,alibi-CF,NaN,NaN,...,NaN,NaN,0.722879,NaN,NaN,4.768510,0.0,0,NaN,1.0
1,1,mnist_output_100,0.0,7,7,NaN,1.0,alibi-CF,NaN,NaN,...,NaN,NaN,0.722879,NaN,NaN,4.600064,0.0,0,NaN,1.0
2,2,mnist_output_100,0.0,7,7,NaN,2.0,alibi-CF,NaN,NaN,...,NaN,NaN,0.722879,NaN,NaN,3.915866,0.0,0,NaN,1.0
3,3,mnist_output_100,0.0,7,7,NaN,3.0,alibi-CF,NaN,NaN,...,NaN,NaN,0.722879,NaN,NaN,3.973385,0.0,0,NaN,1.0
4,4,mnist_output_100,0.0,7,7,NaN,4.0,alibi-CF,NaN,NaN,...,NaN,NaN,0.722879,NaN,NaN,3.924591,0.0,0,NaN,1.0


In [21]:
df = pd.read_csv('cifar_full_results.csv')
print(df.columns)

metrics = ['IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility', 'morans_i_difference','morans_i_explanation', 'optim_time']
data = "CIFAR"

Index(['Unnamed: 0', 'network', 'image', 'original_label',
       'original_predicted_label', 'predicted_label', 'target', 'method',
       'IM1', 'IM2', 'correctness', 'l1', 'l2', 'linf', 'implausibility',
       'morans_i_original', 'morans_i_explanation',
       'morans_i_original_minus_explanation', 'optim_time',
       'correctness_ratio', 'weak_correctness', 'morans_i_difference',
       'timeout'],
      dtype='object')


In [22]:
aggregated = df.groupby('method').agg({
    'IM1': 'mean',
    'IM2': 'mean',
    'l2': 'mean',
    'implausibility': 'mean',
    'morans_i_difference':'mean',
    'morans_i_explanation':'mean',
   'optim_time': 'mean',
   'correctness_ratio':'mean'
})

aggregated['IM2'] = aggregated['IM2'].apply(lambda x: f"{x:.2e}")
aggregated = aggregated.round(2)
print(aggregated.to_latex(float_format="%.2f"))

\begin{tabular}{lrlrrrrrr}
\toprule
 & IM1 & IM2 & l2 & implausibility & morans_i_difference & morans_i_explanation & optim_time & correctness_ratio \\
method &  &  &  &  &  &  &  &  \\
\midrule
C-Min-Edit & 1.05 & 1.68e-03 & 16.16 & 0.01 & 0.05 & 0.86 & 10.62 & 0.99 \\
Captum-MinParamPerturbation & 1.03 & 2.94e-03 & 6.54 & 0.02 & 0.02 & 0.81 & 0.35 & 0.00 \\
Min-Edit & 1.04 & 1.66e-03 & 16.19 & 0.01 & 0.05 & 0.86 & 6.83 & 0.98 \\
PIECE & 1.03 & 1.68e-03 & 16.43 & 0.01 & 0.05 & 0.86 & 16.85 & 0.75 \\
alibi-CF & 1.08 & 8.20e-04 & 32.12 & -0.01 & 0.01 & 0.82 & 107.99 & 0.92 \\
alibi-Proto-CF & 1.02 & 4.03e-03 & 36.70 & -0.01 & 0.35 & 0.50 & 298.58 & 0.66 \\
\bottomrule
\end{tabular}



In [24]:
# Step 1: Aggregate both mean and std
aggregated = df.groupby('method').agg({
    'IM1': ['mean', 'std'],
    'IM2': ['mean', 'std'],
    'l2': ['mean', 'std'],
    'implausibility': ['mean', 'std'],
    'morans_i_difference': ['mean', 'std'],
    'morans_i_explanation': ['mean', 'std'],
    'optim_time': ['mean', 'std'],
    'correctness_ratio': ['mean', 'std']
})

# Step 2: Flatten the MultiIndex columns
aggregated.columns = ['_'.join(col).strip() for col in aggregated.columns.values]

# Step 3: Combine mean and std into one string per column
for col in ['IM1', 'IM2', 'l2', 'implausibility', 'morans_i_difference', 'morans_i_explanation', 'optim_time', 'correctness_ratio']:
    mean_col = f"{col}_mean"
    std_col = f"{col}_std"
    if col == 'IM2':  # format IM2 in scientific notation
        aggregated[col] = aggregated.apply(
            lambda row: f"{row[mean_col]:.2e} ({row[std_col]:.2e})", axis=1
        )
    else:
        aggregated[col] = aggregated.apply(
            lambda row: f"{row[mean_col]:.2f} ({row[std_col]:.2f})", axis=1
        )

# Step 4: Keep only the new formatted columns
aggregated = aggregated[['IM1', 'IM2', 'l2', 'implausibility', 'morans_i_difference', 'morans_i_explanation', 'optim_time', 'correctness_ratio']]

# Print as LaTeX
print(aggregated.to_latex(escape=False))


\begin{tabular}{lllllllll}
\toprule
 & IM1 & IM2 & l2 & implausibility & morans_i_difference & morans_i_explanation & optim_time & correctness_ratio \\
method &  &  &  &  &  &  &  &  \\
\midrule
C-Min-Edit & 1.05 (0.24) & 1.68e-03 (7.76e-04) & 16.16 (5.84) & 0.01 (0.02) & 0.05 (0.04) & 0.86 (0.08) & 10.62 (7.60) & 0.99 (0.03) \\
Captum-MinParamPerturbation & 1.03 (0.21) & 2.94e-03 (1.54e-03) & 6.54 (4.04) & 0.02 (0.02) & 0.02 (0.03) & 0.81 (0.08) & 0.35 (0.30) & 0.00 (0.00) \\
Min-Edit & 1.04 (0.24) & 1.66e-03 (7.69e-04) & 16.19 (5.82) & 0.01 (0.02) & 0.05 (0.04) & 0.86 (0.08) & 6.83 (6.93) & 0.98 (0.06) \\
PIECE & 1.03 (0.21) & 1.68e-03 (7.52e-04) & 16.43 (5.69) & 0.01 (0.02) & 0.05 (0.04) & 0.86 (0.08) & 16.85 (14.59) & 0.75 (0.29) \\
alibi-CF & 1.08 (0.28) & 8.20e-04 (3.19e-04) & 32.12 (6.41) & -0.01 (0.03) & 0.01 (0.04) & 0.82 (0.09) & 107.99 (77.33) & 0.92 (0.18) \\
alibi-Proto-CF & 1.02 (0.16) & 4.03e-03 (4.16e-03) & 36.70 (8.97) & -0.01 (0.02) & 0.35 (0.29) & 0.50 (0.31) & 298.5

In [26]:
# Now filter out incorrect counterfactuals
filtered_df = df[df['correctness'] == 1]

# Step 1: Aggregate both mean and std
aggregated = filtered_df.groupby('method').agg({
    'IM1': ['mean', 'std'],
    'IM2': ['mean', 'std'],
    'l2': ['mean', 'std'],
    'implausibility': ['mean', 'std'],
    'morans_i_difference': ['mean', 'std'],
    'morans_i_explanation': ['mean', 'std'],
    'optim_time': ['mean', 'std'],
    'correctness_ratio': ['mean', 'std']
})

# Step 2: Flatten the MultiIndex columns
aggregated.columns = ['_'.join(col).strip() for col in aggregated.columns.values]

# Step 3: Combine mean and std into one string per column
for col in ['IM1', 'IM2', 'l2', 'implausibility', 'morans_i_difference', 'morans_i_explanation', 'optim_time', 'correctness_ratio']:
    mean_col = f"{col}_mean"
    std_col = f"{col}_std"
    if col == 'IM2':  # format IM2 in scientific notation
        aggregated[col] = aggregated.apply(
            lambda row: f"{row[mean_col]:.2e} ({row[std_col]:.2e})", axis=1
        )
    else:
        aggregated[col] = aggregated.apply(
            lambda row: f"{row[mean_col]:.2f} ({row[std_col]:.2f})", axis=1
        )

# Step 4: Keep only the new formatted columns
aggregated = aggregated[['IM1', 'IM2', 'l2', 'implausibility', 'morans_i_difference','morans_i_explanation', 'optim_time', 'correctness_ratio']]

# Print as LaTeX
print(aggregated.to_latex(escape=False))


\begin{tabular}{lllllllll}
\toprule
 & IM1 & IM2 & l2 & implausibility & morans_i_difference & morans_i_explanation & optim_time & correctness_ratio \\
method &  &  &  &  &  &  &  &  \\
\midrule
C-Min-Edit & 1.05 (0.24) & 1.68e-03 (7.74e-04) & 16.12 (5.77) & 0.01 (0.02) & 0.05 (0.04) & 0.86 (0.08) & 10.47 (7.06) & 0.99 (0.03) \\
Min-Edit & 1.05 (0.24) & 1.66e-03 (7.64e-04) & 16.13 (5.77) & 0.01 (0.02) & 0.05 (0.04) & 0.86 (0.08) & 6.42 (5.98) & 0.99 (0.05) \\
PIECE & 1.04 (0.24) & 1.58e-03 (6.52e-04) & 15.31 (4.66) & 0.01 (0.02) & 0.05 (0.04) & 0.87 (0.07) & 10.10 (9.07) & 0.87 (0.21) \\
alibi-CF & 1.08 (0.28) & 8.20e-04 (3.19e-04) & 32.12 (6.41) & -0.01 (0.03) & 0.01 (0.04) & 0.82 (0.09) & 112.16 (78.70) & 0.95 (0.12) \\
alibi-Proto-CF & 1.02 (0.16) & 4.03e-03 (4.16e-03) & 36.70 (8.97) & -0.01 (0.02) & 0.35 (0.29) & 0.50 (0.31) & 287.69 (91.23) & 0.71 (0.17) \\
\bottomrule
\end{tabular}



In [67]:
#the plot of all plots 
import itertools 

# Get the unique methods from the DataFrame
methods = df['method'].unique()
sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
# Loop over all pairs of methods (combinations)
for method1, method2 in itertools.combinations(methods, 2):
    for arch1, arch2 in itertools.combinations(df['network'].unique(), 2):
        # Filter data for the two methods
        df_method1 = df[(df['method'] == method1) & (df['network'].isin([arch1, arch2]))]
        df_method2 = df[(df['method'] == method2) & (df['network'].isin([arch1, arch2]))]


        # Merge the data on 'image' so we can compare both methods for the same image
        df_combined = pd.merge(df_method1[['image', 'network', 'correctness_ratio']], 
                            df_method2[['image', 'correctness_ratio']], 
                            on='image', 
                            suffixes=('_method1', '_method2'))
        df_counts = (
            df_combined.groupby(['correctness_ratio_method1', 'correctness_ratio_method2', 'network'])
            .size()
            .reset_index(name='count')
        )

        # Create the scatter plot
        plt.figure(figsize=(8, 6))  # Adjust the figure size if needed
        ax = sns.scatterplot(
            data=df_counts, 
            x='correctness_ratio_method1', 
            y='correctness_ratio_method2', 
            hue='network',  # Color points by network
            palette='tab10',  # Choose a color palette
            sizes=(20, 200),  # Adjust point size range
            size='count',
            alpha=0.7,  # Adjust transparency for overlapping points
        )

        max_value = max(df_combined['correctness_ratio_method1'].max(), 
                        df_combined['correctness_ratio_method2'].max())


        # Add plot details
        plt.xlabel(f'correctness ratio ({method1})', fontsize=12)
        plt.ylabel(f'correctness ratio ({method2})', fontsize=12)
        plt.legend(title="Network", fontsize=10, title_fontsize=12)
        plt.grid(True, linestyle='--', alpha=0.6)
        plt.xlim(-0.01, max_value +0.01)
        plt.ylim(-0.01, max_value+0.01)
        plt.plot([0, max_value], [0, max_value], color='red', linestyle='--', linewidth=2)
        sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


        
        # Save the plot (optional)
        plt.savefig(f'figures-xai/{data}/architecture/{data}_architecture{arch1}_{arch2}scatter_{method1}_vs_{method2}.pdf', dpi=300, bbox_inches='tight')
        plt.close()
    # Show the plot





In [ ]:

def boxplot_per_method(df):

    # Create a boxplot for each metric
    for metric in metrics:
        plt.figure(figsize=(12,12))
        sns.set(font_scale = 2)
        sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
        sns.set_palette('deep')

        
        # Create the boxplot
        sns.boxplot(
            data=df,
            x='method',
            y=metric,
            hue = 'method',
            legend = False
        )
        
        # Add titles and labels
        plt.xlabel('Method')
        plt.ylabel(metric)
        plt.xticks(rotation=45, ha='right')
    
        # Save the plot
        plt.savefig(f"figures-xai/{data}/boxplot/{data}_boxplot_{metric}.pdf", dpi=300, bbox_inches='tight')
        plt.close()
boxplot_per_method(df)

In [ ]:
import itertools
def scatter_plot_per_metric(df):
    plt.figure(figsize=(12,12))
    sns.set(font_scale = 2)
    sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
    # Select the relevant columns for the scatterplots
    hues = ['original_label', 'target', 'network', 'method']
    # Create scatterplots for each combination of metrics
    for x_metric, y_metric in itertools.combinations(metrics, 2):
        for hue in hues:

    
            scatter = sns.scatterplot(
                data=df, 
                x=x_metric, 
                y=y_metric, 
                hue=hue,
                palette='deep', 
                alpha=0.8
            )
            
            # Calculate and annotate Spearman correlation
            r, _ = spearmanr(df[x_metric], df[y_metric])
            plt.annotate(
                f"ρ={r:.2f}", 
                xy=(0.05, 0.95), 
                xycoords='axes fraction', 
                fontsize=12, 
                color='red', 
                ha='left', 
                va='top'
            )
            
            # Add plot labels and title
            plt.title(f'{y_metric} vs {x_metric}', fontsize=14)
            plt.xlabel(x_metric)
            plt.ylabel(y_metric)
            plt.legend(fontsize='small', title_fontsize='small')
            # Save the figure to a file
            filename = f"figures-xai/{data}/scatter/{data}_scatter_{y_metric}_vs_{x_metric}_in_{hue}.png"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()


scatter_plot_per_metric(df)


In [ ]:

import itertools
def scatterplot(df):
    plt.figure(figsize=(12,12))
    sns.set(font_scale = 2)
    sns.set_style({'font.family':'serif', 'font.serif':'Times New Roman'})
    sns.set_palette('deep')
    # Select the relevant columns for the scatterplots
    hues = ['original_label', 'target', 'network']
    metrics = ['IM1', 'implausibility']
    # Create scatterplots for each combination of metrics
    for x_method in itertools.combinations(df['method'].unique(), 1):
        x_metric = metrics[0]
        y_metric = metrics[1]
        for hue in hues:
 
    
            ax = scatter = sns.scatterplot(
                data=df[df.method == x_method[0]], 
                x=x_metric, 
                y=y_metric, 
                hue=hue,
                palette='deep', 
                alpha=0.8
            )
            
          
            # Add plot labels and title
            plt.title(f'{y_metric} vs {x_metric}', fontsize=14)
            plt.xlabel(x_metric)
            plt.ylabel(y_metric)
            plt.legend(fontsize='small', title_fontsize='small')
            ax.set(xscale="log", yscale="log")
            sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))


            # Save the figure to a file
            filename = f"figures-xai/{data}/scatter/{data}_scatter_{x_method[0]}_in_{hue}.pdf"
            plt.savefig(filename, dpi=300, bbox_inches='tight')
            plt.close()


scatterplot(df)